In [23]:
import javalang
import json
from tqdm import tqdm
import collections
import sys
import re
from nltk import stem
sn = stem.SnowballStemmer('english')


def get_name(obj):
    if(type(obj).__name__ in ['list', 'tuple']):
        a = []
        for i in obj:
            a.append(get_name(i))
        return a
    elif(type(obj).__name__ in ['dict', 'OrderedDict']):
        a = {}
        for k in obj:
            a[k] = get_name(obj[k])
        return a
    elif(type(obj).__name__ not in ['int', 'float', 'str', 'bool']):
        return type(obj).__name__
    else:
        return obj


def get_info(line):
        code = line.strip()
        tokens = list(javalang.tokenizer.tokenize(code))
        
        code_token=[sn.stem(i.value) for i in tokens]
        
        api=[]
        for i in tokens:
            if i.__class__==javalang.tokenizer.String:
                if sn.stem(i.value) not in api:
                  
                   api.extend(re.sub("[.\"]"," ",sn.stem(i.value)).split())

        tmp=[]

        for i in tokens:
            tmp.append(i)
            if i.value=="(":
                break

        method=[sn.stem(i) for i in re.findall('[A-z][^A-Z]*', tmp[-2].value)]
        
        tks = []
        for tk in tokens:
            if tk.__class__.__name__ == 'String' or tk.__class__.__name__ == 'Character':
                tks.append('STR_')
            elif 'Integer' in tk.__class__.__name__ or 'FloatingPoint' in tk.__class__.__name__:
                tks.append('NUM_')
            elif tk.__class__.__name__ == 'Boolean':
                tks.append('BOOL_')
            else:
                tks.append(tk.value)
        
        
   
        tokens = javalang.tokenizer.tokenize(" ".join(tks).strip())
        token_list = list(javalang.tokenizer.tokenize(" ".join(tks).strip()))
        length = len(token_list)
        parser = javalang.parser.Parser(tokens)
        try:
            tree = parser.parse_member_declaration()
        except (javalang.parser.JavaSyntaxError, IndexError, StopIteration, TypeError):
            return []
        flatten = []
        for path, node in tree:
            flatten.append({'path': path, 'node': node})

        ign = False
        outputs = []
        stop = False
        for i, Node in enumerate(flatten):
            d = collections.OrderedDict()
            path = Node['path']
            node = Node['node']
            children = []
            for child in node.children:
                child_path = None
                if isinstance(child, javalang.ast.Node):
                    child_path = path + tuple((node,))
                    for j in range(i + 1, len(flatten)):
                        if child_path == flatten[j]['path'] and child == flatten[j]['node']:
                            children.append(j)
                if isinstance(child, list) and child:
                    child_path = path + (node, child)
                    for j in range(i + 1, len(flatten)):
                        if child_path == flatten[j]['path']:
                            children.append(j)
            d["id"] = i
            d["type"] = get_name(node)
            if children:
                d["children"] = children
            value = None
            if hasattr(node, 'name'):
                value = node.name
            elif hasattr(node, 'value'):
                value = node.value
            elif hasattr(node, 'position') and node.position:
                for i, token in enumerate(token_list):
                    if node.position == token.position:
                        pos = i + 1
                        value = str(token.value)
                        while (pos < length and token_list[pos].value == '.'):
                            value = value + '.' + token_list[pos + 1].value
                            pos += 2
                        break
            elif type(node) is javalang.tree.This \
                    or type(node) is javalang.tree.ExplicitConstructorInvocation:
                value = 'this'
            elif type(node) is javalang.tree.BreakStatement:
                value = 'break'
            elif type(node) is javalang.tree.ContinueStatement:
                value = 'continue'
            elif type(node) is javalang.tree.TypeArgument:
                value = str(node.pattern_type)
            elif type(node) is javalang.tree.SuperMethodInvocation \
                    or type(node) is javalang.tree.SuperMemberReference:
                value = 'super.' + str(node.member)
            elif type(node) is javalang.tree.Statement \
                    or type(node) is javalang.tree.BlockStatement \
                    or type(node) is javalang.tree.ForControl \
                    or type(node) is javalang.tree.ArrayInitializer \
                    or type(node) is javalang.tree.SwitchStatementCase:
                value = 'None'
            elif type(node) is javalang.tree.VoidClassReference:
                value = 'void.class'
            elif type(node) is javalang.tree.SuperConstructorInvocation:
                value = 'super'

            if value is not None and type(value) is type('str'):
                d['value'] = value
                outputs.append(d["type"]+"."+ d["value"] )
            if not children and not value:
                # print('Leaf has no value!')
                #print(type(node))
                #print(code)
                ign = True
                ign_cnt += 1
                # break
            
        return outputs,method,code_token,api



s='''public void setTypeS(String v) { /*123*/ if (Gold_Type.featOkTst && ((Gold_Type)jcasType).casFeat_typeS == null) jcasType.jcas.throwFeatMissing("typeS", "ch.epfl.bbp.uima.types.Gold"); jcasType.ll_cas.ll_setStringValue(addr, ((Gold_Type)jcasType).casFeatCode_typeS, v);}
'''
out=get_info(s)
print(out)

(['MethodDeclaration.setTypeS', 'FormalParameter.v', 'ReferenceType.String', 'IfStatement.if', 'MemberReference.Gold_Type.featOkTst', 'ReferenceType.Gold_Type', 'MemberReference.jcasType', 'Literal.null', 'StatementExpression.jcasType.jcas.throwFeatMissing', 'MethodInvocation.jcasType.jcas.throwFeatMissing', 'MemberReference.STR_', 'MemberReference.STR_', 'StatementExpression.jcasType.ll_cas.ll_setStringValue', 'MethodInvocation.jcasType.ll_cas.ll_setStringValue', 'MemberReference.addr', 'ReferenceType.Gold_Type', 'MemberReference.jcasType', 'MemberReference.v'], ['set', 'type', 's'], ['public', 'void', 'settyp', '(', 'string', 'v', ')', '{', 'if', '(', 'gold_typ', '.', 'featoktst', '&&', '(', '(', 'gold_typ', ')', 'jcastyp', ')', '.', 'casfeat_typ', '==', 'null', ')', 'jcastyp', '.', 'jcas', '.', 'throwfeatmiss', '(', '"types"', ',', '"ch.epfl.bbp.uima.types.gold"', ')', ';', 'jcastyp', '.', 'll_cas', '.', 'll_setstringvalu', '(', 'addr', ',', '(', '(', 'gold_typ', ')', 'jcastyp', ')'

In [24]:
tokens = list(javalang.tokenizer.tokenize(s))

In [25]:
for i in tokens:
  print(i)

Modifier "public" line 1, position 1
Keyword "void" line 1, position 8
Identifier "setTypeS" line 1, position 13
Separator "(" line 1, position 21
Identifier "String" line 1, position 22
Identifier "v" line 1, position 29
Separator ")" line 1, position 30
Separator "{" line 1, position 32
Keyword "if" line 1, position 42
Separator "(" line 1, position 45
Identifier "Gold_Type" line 1, position 46
Separator "." line 1, position 55
Identifier "featOkTst" line 1, position 56
Operator "&&" line 1, position 66
Separator "(" line 1, position 69
Separator "(" line 1, position 70
Identifier "Gold_Type" line 1, position 71
Separator ")" line 1, position 80
Identifier "jcasType" line 1, position 81
Separator ")" line 1, position 89
Separator "." line 1, position 90
Identifier "casFeat_typeS" line 1, position 91
Operator "==" line 1, position 105
Null "null" line 1, position 108
Separator ")" line 1, position 112
Identifier "jcasType" line 1, position 114
Separator "." line 1, position 122
Identi